# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Malang43/flyrank-ml-internship-Malang43/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Lane: Refresh / Content Opportunity Scoring

ML task type: Ranking / scoring.

The goal is to assign each content page a priority score and rank pages from highest to lowest review priority. A classification model may be used internally to estimate decline probability, but the useful final output is a ranked review queue rather than only a yes/no prediction.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

For the starter dataset, I will use whether a page is currently labelled as declining as a provisional proxy target. The proxy is defined as trend_direction == "down". This is useful for learning the workflow, but it is not yet a true future outcome. A stronger later target could use past data to predict decline in a future time window.

In [3]:
import pandas as pd

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

# Apply the starter-data eligibility conditions
lane_df = df[
    (df["impressions_90d"] > 0) &
    (df["content_age_days"] >= 90)
].drop_duplicates("content_id").copy()

# Provisional target
lane_df["decline_proxy"] = (
    lane_df["trend_direction"] == "down"
).astype(int)

print(lane_df["decline_proxy"].value_counts())
print()
print(lane_df["decline_proxy"].value_counts(normalize=True))

decline_proxy
1    16262
0    13738
Name: count, dtype: int64

decline_proxy
1    0.542067
0    0.457933
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary success metric: Precision@50.

If a reviewer has capacity to inspect only 50 pages, Precision@50 measures how many of the top 50 recommended pages match the selected opportunity proxy. This metric is more useful here than overall accuracy because the real decision is which limited number of pages should be reviewed first.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis: One content page. Each row used for this task should represent one unique content item.

In [4]:
print("Number of rows:", len(lane_df))
print("Unique content pages:", lane_df["content_id"].nunique())

print(
    "One row per content page:",
    len(lane_df) == lane_df["content_id"].nunique()
)

display(
    lane_df[
        [
            "content_id",
            "impressions_90d",
            "sessions_90d",
            "content_age_days",
            "trend_direction",
            "decline_proxy"
        ]
    ].head(10)
)

Number of rows: 30000
Unique content pages: 30000
One row per content page: True


,content_id,impressions_90d,sessions_90d,content_age_days,trend_direction,decline_proxy
0,content_304f48230142,3803,17,187,down,1
1,content_a1fb4e703a9e,15320,9,445,down,1
2,content_9aa793d4d895,12581,11,141,down,1
3,content_331d6c4de07b,11751,78,463,stable,0
4,content_d99b7a2d90ca,19140,145,263,down,1
5,content_d4084a4bc775,3970,5,147,down,1
6,content_9a34b442b552,20,1,90,down,1
7,content_a63219c6e95a,1724,28,445,stable,0
8,content_5e6c160719bc,32574,68,90,down,1
9,content_c27558df2b0c,1240,3,257,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule can prioritize pages using simple thresholds such as age or impressions, but page performance depends on several signals at the same time. ML can learn combinations and interactions among visibility, traffic, age, engagement, and other observable features instead of requiring every threshold to be manually chosen. However, ML should only be preferred if validation shows that it improves the ranked review queue compared with a transparent rule-based baseline.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.